# 🚀 Indic LLM Inference Server & QLoRA Playground (Malayalam QA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bilalvpm4321/malayalam-machine-learning/blob/main/notebooks/indic_llm_inference_server.ipynb)

This notebook serves the **Qwen/Qwen2.5-3B-Instruct** model (with optional QLoRA adapters) as a REST API for the **Indic LLM Evaluation Lab** frontend.

### Steps to Run:
1. Ensure GPU is enabled: **Runtime -> Change runtime type -> T4 GPU**
2. Run all cells: **Runtime -> Run all** (or `Ctrl + F9`)
3. Copy the **Public Tunnel URL** printed at the bottom and paste it into your local frontend dashboard.

### 1. Check GPU Hardware

In [ ]:
!nvidia-smi

### 2. Install Required Packages

In [ ]:
!pip install -q fastapi uvicorn pydantic nest-asyncio pyngrok transformers peft accelerate bitsandbytes sentencepiece

### 3. Load Model and Tokenizer
We use `Qwen/Qwen2.5-3B-Instruct` with 4-bit quantization (or float16) to fit comfortably within the 15GB T4 GPU VRAM.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
print(f"📦 Loading model: {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# 4-bit Quantization Config for fast inference and low memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("✅ Model successfully loaded on GPU!")
print(f"GPU Allocated Memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

### 4. Define FastAPI Server & Endpoints

In [ ]:
import time
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional

app = FastAPI(title="Indic LLM Malayalam QA Server", version="1.0.0")

# Allow CORS so the frontend web app can query the backend
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class PredictionRequest(BaseModel):
    language: str = "ml"
    task: str = "qa"
    configuration: str = "base"
    context: str
    question: str
    expected_answer: Optional[str] = ""

@app.get("/health")
def health_check():
    return {
        "status": "ok",
        "model": MODEL_ID,
        "supported_configurations": ["base", "qlora", "rag"],
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "gpu_memory_used_gb": round(torch.cuda.memory_allocated(0) / 1024**3, 2) if torch.cuda.is_available() else 0,
        "version": "1.0.0"
    }

@app.post("/predict")
def predict(req: PredictionRequest):
    start_time = time.time()
    
    # Malayalam system prompt for strict extractive QA
    system_prompt = (
        "You are a precise Malayalam Question Answering assistant. "
        "Extract the concise answer strictly from the given Malayalam context passage. "
        "Do not make up facts or add commentary outside the context."
    )
    
    user_content = f"Context:\n{req.context}\n\nQuestion:\n{req.question}"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]
    
    # Format with Qwen2.5 chat template
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode response tokens
    response_tokens = output_ids[0][inputs.input_ids.shape[1]:]
    generated_answer = tokenizer.decode(response_tokens, skip_special_tokens=True).strip()
    
    latency_ms = int((time.time() - start_time) * 1000)
    
    # Evaluation metrics
    expected = (req.expected_answer or "").strip()
    exact_match = 1 if (expected and generated_answer == expected) else (0 if expected else None)
    
    # Simple F1 overlap calculation for Indic characters
    f1 = None
    if expected:
        pred_tokens = set(generated_answer.split())
        gold_tokens = set(expected.split())
        overlap = len(pred_tokens & gold_tokens)
        if len(pred_tokens) + len(gold_tokens) > 0:
            f1 = round((2 * overlap) / (len(pred_tokens) + len(gold_tokens)), 4)
        else:
            f1 = 0.0
    
    return {
        "answer": generated_answer,
        "expected_answer": req.expected_answer or None,
        "exact_match": exact_match,
        "f1": f1,
        "similarity": f1 if f1 is not None else (1.0 if exact_match == 1 else 0.0),
        "latency_ms": latency_ms,
        "model": MODEL_ID,
        "configuration": req.configuration,
        "language": req.language
    }

print("✅ FastAPI endpoints created: GET /health, POST /predict")

### 5. Expose with Cloudflare Tunnel or Ngrok and Start Server

We use **Cloudflare Quick Tunnel** (zero account/tokens needed!) with **ngrok** as an automatic fallback.

In [ ]:
import os
import subprocess
import nest_asyncio
import uvicorn
import re

# Download Cloudflare tunnel binary if not present
if not os.path.exists("cloudflared"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# Start cloudflared tunnel in background
log_file = open("tunnel.log", "w")
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"], stdout=log_file, stderr=log_file)

time.sleep(4)

# Read the generated trycloudflare URL
tunnel_url = None
with open("tunnel.log", "r") as f:
    logs = f.read()
    matches = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', logs)
    if matches:
        tunnel_url = matches[0]

print("=" * 65)
if tunnel_url:
    print("🎉 PUBLIC CLOUDFLARE TUNNEL URL:")
    print(f"👉 {tunnel_url}")
    print("\nCopy this URL and paste it into your local Web App Settings!")
else:
    print("Cloudflare tunnel is starting, check tunnel.log if URL hasn't appeared yet.")
print("=" * 65)

# Run Uvicorn server
nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)